In [ ]:
import asyncio
import contextlib
import os
import sys
from pathlib import Path

# Resolve project root no matter where the notebook kernel starts.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "main").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_MAIN = PROJECT_ROOT / "src" / "main"
if str(SRC_MAIN) not in sys.path:
    sys.path.insert(0, str(SRC_MAIN))

# Adds anychat/src/main to sys.path for llms imports outside IntelliJ.
import bootstrap  # noqa: F401

from ingestor.PostIngestor import PostIngestor
from ingestor.StreamClient import StreamClient
from sink.PostgresSink import PostgresSink

N_MINUTES = 1
POSTGRES_DSN = os.getenv("POSTGRES_DSN", "postgresql://localhost/postgres")
CURSOR_FILE = str(PROJECT_ROOT / "data" / "jetstream_cursor.txt")


async def run_ingestion(minutes: int = N_MINUTES) -> None:
    sink = PostgresSink(
        dsn=POSTGRES_DSN,
        batch_size=500,
        flush_interval_s=1.0,
        create_schema=True,
    )
    client = StreamClient(
        [PostIngestor()],
        sink=sink,
        cursor_file=CURSOR_FILE,
        sink_flush_interval_s=1.0,
    )

    task = asyncio.create_task(client.run_forever())
    try:
        await asyncio.sleep(minutes * 60)
    finally:
        task.cancel()
        with contextlib.suppress(asyncio.CancelledError):
            await task
        await sink.close()

    print(f"Ingestion completed for {minutes} minutes.")


await run_ingestion()